# Border-effect sanity checks: image mask vs. filtered-bin border -- Visium HD demo samples

One row per sample (sample name on the left), three columns, one legend for all rows (saved for the supplement as
`result_plots/spatial_omics/ST_sanity_demo.pdf`), all in the same image/mask coordinates
(every 8um bin's position on the mask comes from
`bosperrus.quantify_diffusion_from_raw(..., return_data=True)`, so no
grid-vs-image orientation guessing is needed):

1. **Image + contours** -- the tissue image exactly as it was segmented
   (Visium hires H&E), the image-only tissue mask (pink) -- the border
   for both the buffer and the diffusion fit -- and, for comparison only, the
   footprint of Space Ranger's `in_tissue` bins in qualifying components (green, dotted).
2. **Counts + border/elbow/BLADE** -- the analysed bins (inside the image mask,
   n_counts > 0, `st_precompute.py --image-mask`), the image
   mask (pink), the bosperrus elbow buffer (blue; rising-only piecewise-linear fit
   against the distance to the tissue edge, within 1 mm; solid = p <= 0.05 against the
   interior windows, dashed = rising elbow that fails it) and the BLADE buffer (orange, peeled from
   the analysed bins' own edge), each fit per spatially-connected component.
3. **RAW counts outside the image mask** -- whole capture area (zero-count bins
   included), raw 8um bins outside the mask, i.e. exactly what the diffusion fit sees.

All numbers come from `src/spatial_omics/st_precompute.py` (compute node), stored in
`results/spatial_omics/st_reroll/`; this notebook only plots.

In [ ]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import ndimage

from bosperrus import FIT_PALETTE

import sys
sys.path.insert(0, "../../src/spatial_omics")
from border_criterion import criterion_metrics, load_window_null, passes, window_p_value

RESULTS_DIR = Path("../../results/spatial_omics/st_reroll")  # written by src/spatial_omics/st_precompute.py
SUFFIX = "_imagemask"  # main analysis: image mask, tissue-edge distance, rising-only fit
SAMPLES = pd.read_csv("../../src/spatial_omics/st_samples.csv")
SAMPLES = SAMPLES[SAMPLES["dataset_type"].isin(["visium_demo"])].reset_index(drop=True)
NULL = load_window_null(RESULTS_DIR / "interior_windows", SAMPLES["name"])  # 8 um demo interior windows


In [ ]:
MARGIN_FRAC = 0.05             # column 1/2 crop margin around qualifying bins, fraction of their extent
BSPBLUE = FIT_PALETTE["Piecewise Linear Fit"]
BLADE_ORANGE = "orange"
IMAGE_MASK_COLOR = "#FF1493"    # pink, matches the H&E staining
SPOT_MASK_COLOR = "limegreen"   # dotted; visible on H&E and ssDNA

N_SAMPLES = len(SAMPLES)

In [ ]:
def _pad_linear(a):
    """Pad a 2D coordinate array by one cell on every side, extrapolating linearly."""
    a = np.pad(a, 1, mode="edge").astype(float)
    a[0, :], a[-1, :] = 2 * a[1, :] - a[2, :], 2 * a[-2, :] - a[-3, :]
    a[:, 0], a[:, -1] = 2 * a[:, 1] - a[:, 2], 2 * a[:, -2] - a[:, -3]
    return a


def load_sample(sample):
    """Everything st_precompute.py wrote for one sample (image-mask bins + Space Ranger's for comparison), plus the dense-grid
    mask coordinates of every bin: raw bins cover the full capture-area grid,
    so MASK_ROW[array_row, array_col] / MASK_COL[...] map any grid raster onto
    the image/mask -- contours drawn with these 2D coordinates land in image
    space directly, whatever the grid's orientation/rotation relative to the image."""
    npz = np.load(RESULTS_DIR / f"{sample}_diffusion_8um.npz")
    raw = {k: npz[k] for k in npz.files}
    filtered = pd.read_parquet(RESULTS_DIR / f"{sample}_filtered_bins{SUFFIX}.parquet")  # analysed bins
    spaceranger = pd.read_parquet(RESULTS_DIR / f"{sample}_filtered_bins.parquet",
                                  columns=["array_row", "array_col", "components"])  # comparison only
    shape = (raw["array_row"].max() + 1, raw["array_col"].max() + 1)
    grid_row = np.full(shape, np.nan); grid_col = np.full(shape, np.nan)
    grid_row[raw["array_row"], raw["array_col"]] = raw["mask_row"]
    grid_col[raw["array_row"], raw["array_col"]] = raw["mask_col"]
    if np.isnan(grid_row).any():
        raise ValueError(f"{sample}: raw bins don't cover the full grid")
    return raw, filtered, spaceranger, _pad_linear(grid_row), _pad_linear(grid_col)


def passing_components(sample, filtered):
    """Component labels whose rising elbow has p <= ALPHA against the interior windows."""
    per_component = json.loads((RESULTS_DIR / f"{sample}_buffer{SUFFIX}.json").read_text())["per_component"]
    fitted = filtered[~filtered["virtual"] & np.isfinite(filtered["distance_to_border"])]
    d_max = fitted.groupby("components")["distance_to_border"].max()
    return {int(c) for c, info in per_component.items() if passes(window_p_value(*criterion_metrics(info, d_max[int(c)]), NULL))}


def grid_raster(shape, array_row, array_col, values):
    """Values on the (1-cell padded) capture-area grid, 0 elsewhere."""
    out = np.zeros((shape[0] + 2, shape[1] + 2))
    out[array_row + 1, array_col + 1] = values
    return out

## Build the figure, one row per sample

In [ ]:
def process_sample_row(axes_row, i):
    sample = SAMPLES["name"].iloc[i]
    print("reading", sample)
    raw, filtered, footprint, grid_row, grid_col = load_sample(sample)
    shape = (grid_row.shape[0] - 2, grid_row.shape[1] - 2)
    mask = raw["mask"]
    mask_x, mask_y = np.arange(mask.shape[1]), np.arange(mask.shape[0])

    qual = filtered["components"].to_numpy() >= 0  # bins in a qualifying (>= MIN_COMPONENT_SPOTS) component
    q = filtered[qual]
    fp = footprint[footprint["components"] >= 0]
    footprint_occupancy = grid_raster(shape, fp["array_row"], fp["array_col"], 1.0)
    ok = q["components"].isin(passing_components(sample, filtered)).to_numpy()
    bsp_pass_grid = grid_raster(shape, q["array_row"], q["array_col"], (q["analysis_buffer"].to_numpy() & ok).astype(float))
    bsp_fail_grid = grid_raster(shape, q["array_row"], q["array_col"], (q["analysis_buffer"].to_numpy() & ~ok).astype(float))
    blade_grid = grid_raster(shape, q["array_row"], q["array_col"], q["in_blade_buffer"].astype(float))

    # crop (columns 1 and 2) around the qualifying bins, in mask/image coordinates
    r0, r1 = q["mask_row"].min(), q["mask_row"].max()
    c0, c1 = q["mask_col"].min(), q["mask_col"].max()
    mr, mc = (r1 - r0) * MARGIN_FRAC, (c1 - c0) * MARGIN_FRAC
    crop_x, crop_y = (c0 - mc, c1 + mc), (r1 + mr, r0 - mr)  # y inverted: image orientation

    # === Column 1: image + image mask + spot-derived footprint ===
    ax = axes_row[0]
    f = int(raw["image_factor"])
    img = raw["image"]
    ax.imshow(img, cmap="gray" if img.ndim == 2 else None,
              extent=(-0.5, img.shape[1] * f - 0.5, img.shape[0] * f - 0.5, -0.5), rasterized=True)
    ax.contour(mask_x, mask_y, mask.astype(float), levels=[0.5], colors=[IMAGE_MASK_COLOR], linewidths=1.0, rasterized=True)
    ax.contour(grid_col, grid_row, footprint_occupancy, levels=[0.5], colors=[SPOT_MASK_COLOR], linewidths=1.2,
               linestyles=":", rasterized=True)
    ax.set_xlim(*crop_x); ax.set_ylim(*crop_y)
    ax.set_aspect("equal"); ax.axis("off")
    ax.text(-0.03, 0.5, sample, rotation=90, transform=ax.transAxes, ha="right", va="center", fontsize=9)

    # === Column 2: counts inside the image mask + bosperrus elbow + BLADE ===
    ax = axes_row[1]
    ax.scatter(q["mask_col"], q["mask_row"], c=q["n_counts"], cmap="gray", s=1, alpha=0.5, rasterized=True)
    ax.contour(grid_col, grid_row, blade_grid, levels=[0.5], colors=[BLADE_ORANGE], linewidths=1.0, rasterized=True)  # below the elbow
    ax.contour(grid_col, grid_row, bsp_pass_grid, levels=[0.5], colors=[BSPBLUE], linewidths=1.0, rasterized=True)
    if bsp_fail_grid.any():
        ax.contour(grid_col, grid_row, bsp_fail_grid, levels=[0.5], colors=[BSPBLUE], linewidths=1.0, linestyles="--", rasterized=True)
    ax.contour(mask_x, mask_y, mask.astype(float), levels=[0.5], colors=[IMAGE_MASK_COLOR], linewidths=0.8, rasterized=True)
    ax.set_xlim(*crop_x); ax.set_ylim(*crop_y)
    ax.set_aspect("equal"); ax.axis("off")

    # === Column 3: RAW counts outside the image mask, whole capture area ===
    ax = axes_row[2]
    outside = raw["distance_um"] > 0
    counts_outside = np.full(shape, np.nan)
    counts_outside[raw["array_row"][outside], raw["array_col"][outside]] = raw["n_counts"][outside]
    vmax = np.percentile(raw["n_counts"][outside], 99)
    mesh = ax.pcolormesh(grid_col[1:-1, 1:-1], grid_row[1:-1, 1:-1], np.ma.masked_invalid(counts_outside),
                         cmap="viridis", vmin=0, vmax=vmax, shading="nearest", rasterized=True)
    ax.contour(mask_x, mask_y, mask.astype(float), levels=[0.5], colors=[IMAGE_MASK_COLOR], linewidths=0.8, rasterized=True)
    cbar = plt.colorbar(mesh, cax=ax.inset_axes([1.03, 0.15, 0.04, 0.7]))
    cbar.ax.tick_params(labelsize=7)
    ax.set_ylim(np.nanmax(grid_row), np.nanmin(grid_row))
    ax.set_xlim(np.nanmin(grid_col), np.nanmax(grid_col))
    ax.set_aspect("equal"); ax.axis("off")

    del raw, filtered, footprint, grid_row, grid_col, mask
    gc.collect()


In [ ]:
fig, axes = plt.subplots(N_SAMPLES, 3, figsize=(8, 2.2 * N_SAMPLES), squeeze=False, layout="constrained")
for i in range(N_SAMPLES):
    process_sample_row(axes[i], i)
for ax, title in zip(axes[0], ["H&E + tissue masks",
                               "counts inside image mask,\nbosperrus elbow and BLADE buffer",
                               "raw counts outside image mask\n(per 8 µm bin, clipped at p99)"]):
    ax.set_title(title, fontsize=9)
fig.legend(handles=[
    Line2D([0], [0], color=IMAGE_MASK_COLOR, lw=1.2, label="image-derived tissue mask"),
    Line2D([0], [0], color=SPOT_MASK_COLOR, lw=1.2, ls=":", label="spot-derived mask (Space Ranger in_tissue)"),
    Line2D([0], [0], color=BSPBLUE, lw=1.2, label="bosperrus elbow (p ≤ 0.05)"),
    Line2D([0], [0], color=BSPBLUE, lw=1.2, ls="--", label="bosperrus elbow (p > 0.05)"),
    Line2D([0], [0], color=BLADE_ORANGE, lw=1.2, label="BLADE buffer"),
], loc="outside upper center", ncol=3, fontsize=8, frameon=False)
Path("../../result_plots/spatial_omics").mkdir(parents=True, exist_ok=True)
plt.savefig("../../result_plots/spatial_omics/ST_sanity_demo.pdf", bbox_inches="tight", dpi=300)  # images, count maps and mask contours are rasterized (vector contours: >10 MB)
plt.show()


## Count vs. distance profiles to the tissue edge, per component

Analysed bins of the tissue-edge run (`st_precompute.py <sample> --image-mask`: raw 8 µm bins
inside the image mask, n_counts > 0, holes filled), distance to the edge of the whole image mask (uncapped,
recomputed from the npz). `bosperrus.plot_fit` per 16 µm distance step: mean (black line), 25-75% and 5-95% bands,
up to 2 mm; the rising-only piecewise-linear fit (fitted on bins within 1 mm, dotted line) in red; the elbow
in blue, solid if p <= 0.05 against the interior windows (`src/spatial_omics/border_criterion.py`),
dashed if p > 0.05.
All pages: `result_plots/spatial_omics/ST_profiles_tissue_edge.pdf`.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from bosperrus import PiecewiseLinearFit, plot_fit
from matplotlib.patches import Patch
from st_precompute import TISSUE_FIT_MAX_DISTANCE_UM, distance_to_tissue_edge

PROFILE_MAX_UM, PROFILE_STEP_UM, N_COLS = 2000.0, 16.0, 6


def component_profiles(sample):
    """Per analysed bin: component, n_counts, uncapped distance to the tissue edge."""
    raw = dict(np.load(RESULTS_DIR / f"{sample}_diffusion_8um.npz"))
    inside = raw["distance_um"] == 0
    full = pd.Series(distance_to_tissue_edge(raw, inside),
                     index=pd.MultiIndex.from_arrays([raw["array_row"][inside], raw["array_col"][inside]]))
    bins = pd.read_parquet(RESULTS_DIR / f"{sample}_filtered_bins{SUFFIX}.parquet",
                           columns=["array_row", "array_col", "n_counts", "components", "virtual", "distance_to_border"])
    bins = bins[~bins["virtual"] & (bins["components"] >= 0)].copy()
    bins["distance_um"] = full.reindex(pd.MultiIndex.from_arrays([bins["array_row"], bins["array_col"]])).to_numpy()
    return bins


def plot_component(ax, g, info, label):
    edges = np.arange(0, min(g["distance_um"].max(), PROFILE_MAX_UM) + PROFILE_STEP_UM, PROFILE_STEP_UM)
    fitted = g[np.isfinite(g["distance_to_border"])]
    eff, half = criterion_metrics(info, fitted["distance_to_border"].max())
    p_value = window_p_value(eff, half, NULL)
    ok = passes(p_value)
    p = info["params"]
    if "piecewise_linear_m" in p:
        predict = lambda d: PiecewiseLinearFit.piecewise_plateau(d, p["piecewise_linear_b"], p["piecewise_linear_m"],
                                                                  p["piecewise_linear_c"])
        verdict = (f"effect {eff:.2f}, half-life {half:.0f} um, p = {p_value:.3f}" if info["border_effect"]
                   else "not rising: fail")
    else:
        predict = lambda d: np.full_like(d, p["constant_c"])
        verdict = "constant: fail"
    # quantile bands + per-bin mean up to PROFILE_MAX_UM; the fit drawn only over its 1 mm fit range
    plot_fit(ax, g["distance_um"], g["n_counts"], predict, bins=edges,
             d_grid=np.linspace(0, min(TISSUE_FIT_MAX_DISTANCE_UM, fitted["distance_to_border"].max()), 200),
             line_kwargs={"color": "red", "linewidth": 1.2})
    ax.set_xlabel(""); ax.set_ylabel("")
    if info["border_effect"]:
        ax.axvline(p["piecewise_linear_b"], color=BSPBLUE, ls="-" if ok else "--", lw=1)
    ax.axvline(TISSUE_FIT_MAX_DISTANCE_UM, color="gray", ls=":", lw=0.8)
    ax.set_xlim(0, edges[-1])
    ax.set_ylim(bottom=0)
    ax.set_title(f"{label} (n={len(g):,})\n{verdict}", fontsize=8, color="k" if ok else "firebrick")
    ax.tick_params(labelsize=7)


Path("../../result_plots/spatial_omics").mkdir(parents=True, exist_ok=True)
with PdfPages("../../result_plots/spatial_omics/ST_profiles_tissue_edge.pdf") as pdf:
    for sample in SAMPLES["name"]:
        bins = component_profiles(sample)
        per_component = json.loads((RESULTS_DIR / f"{sample}_buffer{SUFFIX}.json").read_text())["per_component"]
        labels = sorted(int(c) for c in per_component)
        n_cols = min(N_COLS, len(labels))
        n_rows = int(np.ceil(len(labels) / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 2.6 * n_rows + 0.6), squeeze=False)
        for ax in axes.flat[len(labels):]:
            ax.set_visible(False)
        for ax, c in zip(axes.flat, labels):
            plot_component(ax, bins[bins["components"] == c], per_component[str(c)], f"{sample}_{c}")
        for ax in axes[-1]:
            ax.set_xlabel("distance to tissue edge (um)", fontsize=8)
        for ax in axes[:, 0]:
            ax.set_ylabel("n_counts per 8 um bin", fontsize=8)
        fig.suptitle(sample, fontsize=11)
        plt.tight_layout(rect=(0, 0, 1, 0.97 - 0.25 / fig.get_figheight()))
        fig.legend(handles=[
            Line2D([0], [0], color="black", lw=1, label="mean per 16 µm"),
            Patch(color="0.7", label="25-75%"), Patch(color="0.88", label="5-95%"),
            Line2D([0], [0], color="red", lw=1.2, label="fit (within 1 mm)"),
            Line2D([0], [0], color=BSPBLUE, lw=1, label="elbow (p ≤ 0.05)"),
            Line2D([0], [0], color=BSPBLUE, lw=1, ls="--", label="elbow (p > 0.05)"),
        ], loc="upper center", bbox_to_anchor=(0.5, 0.97 - 0.1 / fig.get_figheight()), ncol=6, fontsize=8, frameon=False)
        pdf.savefig(fig)
        plt.show()
        del bins
